# Lab 01: RNN Sequence Modeling

**Module 01** | Read `notes/01-rnns-sequence-modeling.pdf` before running this notebook.

Train a character-level RNN on a small corpus. You will build the vocabulary, unroll sequences, run a full training loop, and plot the loss curve.

Run every cell top to bottom. Code is complete, no fill-in sections. Markdown cells explain what each block does and why.


## Runtime device

PyTorch can run on your NVIDIA GPU through CUDA or fall back to CPU. GPU execution moves matrix operations off the host and typically speeds up neural network training by a large factor. The next cell detects what is available and prints the active device so you know whether to expect fast training or should keep batch sizes small.


In [ ]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"CUDA enabled, {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available, using CPU. Labs still run; training may take longer.")


Character-level language modeling is the simplest setting for sequence models: each timestep is one symbol, so you can watch the network learn local patterns without tokenizer complexity. We use a short repeating corpus so training finishes quickly on CPU or GPU while still producing a visible loss drop.


The goal is **next-character prediction**. Given 32 consecutive characters, the model predicts the character that follows. That framing matches how autoregressive text models are trained at scale: predict one token at a time, conditioned on everything seen so far.


### Step 1: Build the corpus and vocabulary

The corpus is fixed below. From it we derive a vocabulary (unique characters plus an index mapping) and sliding windows of length `SEQ_LEN`. Each window becomes one training example; the target is the single character immediately after the window.

Keeping batch construction explicit makes the link between raw text and tensor batches clear before any neural network code runs.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

ROOT = Path("..").resolve()
SEQ_LEN = 32
BATCH_SIZE = 64
HIDDEN_SIZE = 128
EPOCHS = 5
LR = 1e-2

CORPUS = "sequence modeling practice text " * 50
chars = sorted(set(CORPUS))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for ch, i in char_to_idx.items()}

indices = torch.tensor([char_to_idx[c] for c in CORPUS], dtype=torch.long)
inputs, targets = [], []
for start in range(len(indices) - SEQ_LEN):
    inputs.append(indices[start : start + SEQ_LEN])
    targets.append(indices[start + SEQ_LEN])
inputs = torch.stack(inputs)
targets = torch.stack(targets)

dataset = TensorDataset(inputs, targets)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"Corpus length: {len(CORPUS):,} chars")
print(f"Vocabulary size: {vocab_size}")
print(f"Training examples: {len(dataset):,}")
print(f"\nVocabulary ({vocab_size} unique chars):")
print("  " + "".join(chars))


### Step 2: Inspect training examples as readable strings

Each row of `inputs` is 32 character indices; `targets` holds the next character index. Decoding a few rows shows exactly what the model sees and what it must predict.


In [ ]:
def decode_indices(idxs: torch.Tensor) -> str:
    return "".join(idx_to_char[i.item()] for i in idxs)


print("Three sample training pairs (input window -> target):")
print("-" * 60)
for i in range(3):
    window = decode_indices(inputs[i])
    target_ch = idx_to_char[targets[i].item()]
    print(f"  [{i}] input:  {window!r}")
    print(f"      target: {target_ch!r}")
    print()


### Step 3: Define the CharRNN

`CharRNN` stacks three standard pieces: an embedding turns discrete character IDs into dense vectors; an `nn.RNN` maintains a hidden state while reading the sequence; a linear layer maps the final hidden state to logits over the vocabulary.

We use the built-in RNN cell rather than a manual loop so you can focus on data flow and training. The same pattern extends to LSTM and GRU in the next lab.


In [ ]:
class CharRNN(nn.Module):
    def __init__(self, vocab_size: int, hidden_size: int = 128):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, hidden_size)
        self.rnn = nn.RNN(hidden_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        embedded = self.embed(x)
        output, _ = self.rnn(embedded)
        logits = self.fc(output[:, -1, :])
        return logits


model = CharRNN(vocab_size, HIDDEN_SIZE).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nParameters: {n_params:,}")


### Step 4: One forward pass before training

Before optimizing weights, run a single example through the model. Logits are raw scores over the vocabulary; `argmax` picks the most likely next character. We also show the top-3 candidates so you can see how uncertain the untrained model is.


In [ ]:
@torch.no_grad()
def show_prediction(model, input_seq: torch.Tensor, target_idx: int) -> None:
    model.eval()
    logits = model(input_seq.unsqueeze(0).to(device))
    probs = torch.softmax(logits, dim=-1).squeeze(0)
    pred_idx = logits.argmax(dim=-1).item()
    top3_probs, top3_idx = probs.topk(3)
    print(f"  Input:    {decode_indices(input_seq)!r}")
    print(f"  Target:   {idx_to_char[target_idx]!r}")
    print(f"  Predicted:{idx_to_char[pred_idx]!r}  {'correct' if pred_idx == target_idx else 'wrong'}")
    print("  Top-3 next-char candidates:")
    for rank, (prob, idx) in enumerate(zip(top3_probs, top3_idx), start=1):
        ch = idx_to_char[idx.item()]
        print(f"    {rank}. {ch!r}  (prob {prob.item():.4f})")


print("Untrained model, one example:")
show_prediction(model, inputs[0], targets[0].item())


### Step 5: Training loop

Training minimizes cross-entropy between predicted logits and the true next character. We track both average loss and **character accuracy** (fraction of examples where `argmax` matches the target) each epoch.

Shuffling batches each epoch prevents the optimizer from memorizing example order, which matters even on small corpora.


In [ ]:
def epoch_accuracy(model, inputs_t: torch.Tensor, targets_t: torch.Tensor, batch_size: int = 256) -> float:
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for start in range(0, len(inputs_t), batch_size):
            batch_x = inputs_t[start : start + batch_size].to(device)
            batch_y = targets_t[start : start + batch_size].to(device)
            preds = model(batch_x).argmax(dim=-1)
            correct += (preds == batch_y).sum().item()
            total += batch_y.size(0)
    return correct / total


loss_history = []
acc_history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for batch_x, batch_y in loader:
        batch_x = batch_x.to(device)
        batch_y = batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * batch_x.size(0)

    avg_loss = epoch_loss / len(dataset)
    acc = epoch_accuracy(model, inputs, targets)
    loss_history.append(avg_loss)
    acc_history.append(acc)
    print(f"Epoch {epoch:02d}/{EPOCHS}  loss {avg_loss:.4f}  char accuracy {acc * 100:.1f}%")


### Step 6: Plot the loss curve

A falling loss curve confirms the RNN is fitting the local statistics of the corpus. If loss flatlines immediately, check learning rate, device placement, or label alignment.


In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOCHS + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("CharRNN training loss")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Step 7: Post-training evaluation on held examples

After training, inspect predictions on a few windows the model did not necessarily see in the last batch order. Accuracy on the full dataset should be well above the random baseline of `1 / vocab_size`.


In [ ]:
print("Post-training predictions on 5 examples:")
print("-" * 60)
for i in [0, 100, 500, 1000, 2000]:
    show_prediction(model, inputs[i], targets[i].item())
    print()

final_acc = epoch_accuracy(model, inputs, targets)
random_baseline = 1.0 / vocab_size
print(f"Full-dataset char accuracy: {final_acc * 100:.2f}%")
print(f"Random-guess baseline:      {random_baseline * 100:.2f}%")


### Step 8: Autoregressive text generation

Language models are trained to predict one character at a time. At inference we feed the model its own predictions: take the seed string, predict the next char, append it, slide the window, and repeat. This is how ChatGPT-style models generate text, one token at a time.


In [ ]:
@torch.no_grad()
def generate_text(model, seed: str, n_chars: int = 80) -> str:
    model.eval()
    text = seed
    for _ in range(n_chars):
        window = text[-SEQ_LEN:]
        idxs = torch.tensor([[char_to_idx[c] for c in window]], device=device)
        logits = model(idxs)
        next_idx = logits.argmax(dim=-1).item()
        text += idx_to_char[next_idx]
    return text


seed = "sequence mod"
generated = generate_text(model, seed, n_chars=80)
print(f"Seed ({len(seed)} chars): {seed!r}")
print(f"Generated ({len(generated)} chars):")
print(generated)


### Summary

You built a character-level RNN, inspected raw training pairs, watched loss and accuracy improve epoch by epoch, and generated new text autoregressively. The repeating corpus makes patterns easy to learn; on real text, vocabulary size and sequence length grow, but the training objective stays the same.


In [ ]:
print("=" * 60)
print("Lab 01 summary")
print("=" * 60)
print(f"  Corpus length:     {len(CORPUS):,} chars")
print(f"  Vocabulary:        {vocab_size} unique characters")
print(f"  Training examples: {len(dataset):,}")
print(f"  Model parameters:  {n_params:,}")
print(f"  Final loss:        {loss_history[-1]:.4f}")
print(f"  Final accuracy:    {acc_history[-1] * 100:.2f}%")
print(f"  Seed:              {seed!r}")
print(f"  Generated length:  {len(generated)} chars")
print("=" * 60)
